In [ ]:
cd(joinpath(@__DIR__, ".."))   # run from the repository root

In [ ]:
using VoronoiMeshes, NCDatasets, DelaunayTriangulation, CairoMakie
include("src/juliaDEC.jl");

In [ ]:
edge_idx = 14
mesh = VoronoiMesh("meshes/mesh3_regular.nc")
xp = mesh.x_period
yp = mesh.y_period


v = discretize_vector_field_dual(mesh, x -> [cos(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp), sin(2*pi*x[1]/xp) * cos(2*pi*x[2]/yp)])
v_dual = discretize_vector_field_primal(mesh, x -> [-sin(2*pi*x[1]/xp) * cos(2*pi*x[2]/yp), cos(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)])
rec_field, coefs, _ = rec_vector_field_lsq_13point(mesh, v, edge_idx , verbose=true, primal_to_dual=true, hodge=false, only_linear=false)
println("Interpolation coefs: ", coefs)
rec_values = integrate_lsq_reconstruction(mesh, v, true, false)

# Checking for the error of the interpolation
# println(maximum(abs.(rec_values.values - v_dual.values)))

println("\nError of LSQ Hodge:")
println(maximum(abs.(hodge_star_lsq_matrix(mesh, true, false) * v.values - v_dual.values)))

println("\nValue of primal cochain in edge $edge_idx: ", v.values[edge_idx])
println("\nValue of dual cochain in edge $edge_idx: ", v_dual.values[edge_idx])

f = Figure(size = (1000, 700))
ax = Axis(f[1, 1])


xs = LinRange(-0.1, 1.1, 30)
ys = LinRange(-0.1, 1.1, 30)
us = [cos(2*pi*x/xp) * sin(2*pi*y/yp) for x in xs, y in ys]
vs = [sin(2*pi*x/xp) * cos(2*pi*y/yp) for x in xs, y in ys]
arrows!(xs, ys, us, vs,arrowsize = 10, lengthscale = 0.04)
neighbors = get_neighbors_3level_primal(mesh, edge_idx)
for neighbor in neighbors
  if neighbor != edge_idx
      scatter!(ax, [mesh.edges.position[neighbor][1]], [mesh.edges.position[neighbor][2]], color="red")
  end
  scatter!(ax, [mesh.edges.position[edge_idx][1]], [mesh.edges.position[edge_idx][2]], color="blue")
end


ax = Axis(f[1, 2])
us = [rec_field([x, y])[1] for x in xs, y in ys]
vs = [rec_field([x, y])[2] for x in xs, y in ys]
arrows!(xs, ys, us, vs, arrowsize = 10, lengthscale = 0.04)
xlims!(ax, (-0.1, 1.1))
ylims!(ax, (-0.1, 1.1))

neighbors = get_neighbors_3level_primal(mesh, edge_idx)
for neighbor in neighbors
  if neighbor != edge_idx
      scatter!(ax, [mesh.edges.position[neighbor][1]], [mesh.edges.position[neighbor][2]], color="red")
  end
  scatter!(ax, [mesh.edges.position[edge_idx][1]], [mesh.edges.position[edge_idx][2]], color="blue")
end
f


### Hodge consistency - Diagonal Hodge Star Operator

In [ ]:
error_list_Linf = []
edge_length_list = []
for i = 1:7
    
    mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    
    println("Testing diagonal hodge star operator")
    xp = mesh.x_period
    yp = mesh.y_period
    A = star(mesh, 1, true)

    field = x-> [cos(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp), sin(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp)]
    hodge_field = x -> [-sin(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp), cos(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp)]
    v = discretize_vector_field_dual(mesh, field)
    v_hodge = discretize_vector_field_primal(mesh, hodge_field)

    #Inf norm
    push!(error_list_Linf, maximum(abs.(A * v.values - v_hodge.values)))

    #L2 norm
    #push!(error_list_Linf, L2_errorOnEdges(mesh, A * v.values, v_hodge.values))
    
    push!(edge_length_list, maximum(sqrt.(mesh.vertices.area)))
    println("number of triangles: ", mesh.vertices.n, " number of edges: ", mesh.edges.n, " number of vertices: ", mesh.cells.n)
    println("The hodge matrix is symmetric: ", issymmetric(A))
    println("The hodge matrix is positive definite: ", isposdef(A))
    println("---------")
end
println(error_list_Linf)
println(edge_length_list)

for i = 2:length(error_list_Linf)
    order = log(error_list_Linf[i]/error_list_Linf[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end



### Hodge Consistency - Whitney Hodge Star Operator

In [ ]:
error_list_Linf = []
edge_length_list = []
for i = 1:7
    
    mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    
    println("Testing whitney hodge star operator")
    xp = mesh.x_period
    yp = mesh.y_period
    A = whitney_hodge_star_matrix(mesh)

    field = x -> [cos(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp), sin(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp)]
    hodge_field = x -> [-sin(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp), cos(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp)]
    
    v = discretize_vector_field_dual(mesh, field)
    v_hodge = discretize_vector_field_primal(mesh, hodge_field)

    #Inf norm
    push!(error_list_Linf, maximum(abs.(A * v.values - v_hodge.values)))

    #L2 norm
    #aux = (A * v.values - v_hodge.values) .^2
    #aux = aux .* (mesh.edges.lengthDual/mesh.edges.length)
    #push!(error_list_Linf, sqrt(sum(aux)))
    
    push!(edge_length_list, maximum(sqrt.(mesh.vertices.area)))
    println("number of triangles: ", mesh.vertices.n, " number of edges: ", mesh.edges.n, " number of vertices: ", mesh.cells.n)
    println("The hodge matrix is symmetric: ", issymmetric(A))
    println("The hodge matrix is positive definite: ", isposdef(A))
    println("---------")
end
println(error_list_Linf)
println(edge_length_list)

for i = 2:length(error_list_Linf)
    order = log(error_list_Linf[i]/error_list_Linf[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end

### Hodge consistency - Least Squares Hodge Star Operator

In [ ]:
error_list_Linf = []
edge_length_list = []
for i = 1:7
    
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    
    println("Testing least squares hodge star operator")
    xp = mesh.x_period
    yp = mesh.y_period
    A = hodge_star_lsq_matrix(mesh, true, true) #quadratic field

    #field = x -> [cos(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp), sin(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp)]
    #hodge_field = x -> [-sin(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp), cos(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp)]
    
    field = x -> [x[1]*sin(x[2]), 0.0]
    hodge_field = x -> [0.0, x[1]*sin(x[1])]
    v = discretize_vector_field_dual(mesh, field)
    v_hodge = discretize_vector_field_primal(mesh, hodge_field)

    #Inf norm
    push!(error_list_Linf, maximum(abs.(A * v.values - v_hodge.values)))

    #L2 norm
    #aux = (A * v.values - v_hodge.values) .^2
    #aux = aux .* (mesh.edges.lengthDual/mesh.edges.length)
    #push!(error_list_Linf, sqrt(sum(aux)))
    
    push!(edge_length_list, maximum(sqrt.(mesh.vertices.area)))
    println("number of triangles: ", mesh.vertices.n, " number of edges: ", mesh.edges.n, " number of vertices: ", mesh.cells.n)
    println("The hodge matrix is symmetric: ", issymmetric(A))
    println("The hodge matrix is positive definite: ", isposdef(A))
    println("---------")
end
println(error_list_Linf)
println(edge_length_list)

for i = 2:length(error_list_Linf)
    order = log(error_list_Linf[i]/error_list_Linf[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end

## Laplacian Consistency with Diagonal Hodge Star Operator

In [ ]:
function laplacian_diagonal(mesh::VoronoiMesh, u::Cochain{T}) where T
    if u.k != 0 || !u.is_primal
        error("Input must be a primal 0-form")
    end
    # First d: 0-form -> 1-form on triangle edges
    du = d(mesh, u)
    
    # First Hodge: 1-form on triangle edges -> 1-form on hexagon edges
    hodge_du = Cochain(star(mesh, 1, true) * du.values, 1, false)
    # Second d: 1-form on hexagon edges -> 2-form on hexagon cells
    d_hodge_du = d(mesh, hodge_du)
    
    #Hodge: 2-form on hexagon cells -> 0-form on triangle vertices
    return Cochain(star(mesh, 2, false) * d_hodge_du.values, 0, true)
end


error_list_diagonal = []
edge_length_list = []
#for i=1:6
for i = 1:7

mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
#mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    println("i = ", i)
    println("nCells: ", mesh.cells.n)
    println("nEdges: ", mesh.edges.n)
    println("nVertices: ", mesh.vertices.n)
    xp = mesh.x_period
    yp = mesh.y_period
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    u = x -> sin(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)
    true_laplacian = x -> -8*(pi^2)*sin(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)

    u_cochain = Cochain(u.(mesh.cells.position), 0, true)
    lap_cochain = Cochain(true_laplacian.(mesh.cells.position), 0, true)

    lap_diagonal = laplacian_diagonal(mesh, u_cochain)

    push!(error_list_diagonal, maximum(abs.(lap_diagonal.values - lap_cochain.values)))
    #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
    push!(edge_length_list, maximum(mesh.edges.lengthDual))
    println("--------------------------------")
end

println("Error list diagonal: ", error_list_diagonal)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_diagonal)
    order = log(error_list_diagonal[i]/error_list_diagonal[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end

In [ ]:
function laplacian_lsq(mesh::VoronoiMesh, u::Cochain{T}) where T
    if u.k != 0 || !u.is_primal
        error("Input must be a primal 0-form")
    end
    # First d: 0-form -> 1-form on triangle edges
    du = d(mesh, u)
    
    # First Hodge: 1-form on triangle edges -> 1-form on hexagon edges
    hodge_du = Cochain(whitney_hodge_star_matrix(mesh) * du.values, 1, false)
    # Second d: 1-form on hexagon edges -> 2-form on hexagon cells
    d_hodge_du = d(mesh, hodge_du)
    
    #Hodge: 2-form on hexagon cells -> 0-form on triangle vertices
    return Cochain(star(mesh, 2, false) * d_hodge_du.values, 0, true)
end


error_list_lsq = []
edge_length_list = []
#for i=1:6
for i = 1:5

    mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    println("i = ", i)
    println("nCells: ", mesh.cells.n)
    println("nEdges: ", mesh.edges.n)
    println("nVertices: ", mesh.vertices.n)
    xp = mesh.x_period
    yp = mesh.y_period
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    u = x -> sin(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)
    true_laplacian = x -> -8*(pi^2)*sin(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)

    u_cochain = Cochain(u.(mesh.cells.position), 0, true)
    lap_cochain = Cochain(true_laplacian.(mesh.cells.position), 0, true)

    lap_lsq = laplacian_lsq(mesh, u_cochain)

    push!(error_list_lsq, maximum(abs.(lap_lsq.values - lap_cochain.values)))
    #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
    push!(edge_length_list, maximum(mesh.edges.lengthDual))
    println("--------------------------------")
end

println("Error list diagonal: ", error_list_lsq)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_lsq)
    order = log(error_list_lsq[i]/error_list_lsq[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end

## Laplacian Consistency with Least Squares Hodge Star Operator

In [ ]:
function laplacian_lsq(mesh::VoronoiMesh, u::Cochain{T}) where T
    if u.k != 0 || !u.is_primal
        error("Input must be a primal 0-form")
    end
    # First d: 0-form -> 1-form on triangle edges
    du = d(mesh, u)
    
    # First Hodge: 1-form on triangle edges -> 1-form on hexagon edges
    hodge_du = Cochain(hodge_star_lsq_matrix(mesh, true, false) * du.values, 1, false)
    # Second d: 1-form on hexagon edges -> 2-form on hexagon cells
    d_hodge_du = d(mesh, hodge_du)
    
    #Hodge: 2-form on hexagon cells -> 0-form on triangle vertices
    return Cochain(star(mesh, 2, false) * d_hodge_du.values, 0, true)
end


error_list_lsq = []
edge_length_list = []
#for i=1:6
for i = 1:5

    mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    println("i = ", i)
    println("nCells: ", mesh.cells.n)
    println("nEdges: ", mesh.edges.n)
    println("nVertices: ", mesh.vertices.n)
    xp = mesh.x_period
    yp = mesh.y_period
    #mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    u = x -> sin(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)
    true_laplacian = x -> -8*(pi^2)*sin(2*pi*x[1]/xp) * sin(2*pi*x[2]/yp)

    u_cochain = Cochain(u.(mesh.cells.position), 0, true)
    lap_cochain = Cochain(true_laplacian.(mesh.cells.position), 0, true)

    lap_lsq = laplacian_lsq(mesh, u_cochain)

    push!(error_list_lsq, maximum(abs.(lap_lsq.values - lap_cochain.values)))
    #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
    push!(edge_length_list, maximum(mesh.edges.lengthDual))
    println("--------------------------------")
end

println("Error list diagonal: ", error_list_lsq)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_lsq)
    order = log(error_list_lsq[i]/error_list_lsq[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end


### Teste com o Divergente

### Hodge Star Diagonal - Malha regular


In [ ]:
function divergence_diagonal(mesh::VoronoiMesh, u::Cochain{T}) where T
    if u.k != 1 || !u.is_primal
        error("Input must be a primal 1-form")
    end
    d1 = d1_matrix_dual(mesh)
    star_u = star(mesh, 1, true)
    return Cochain(d1 * star_u * u.values, 2, false)
end

error_list_diagonal = []
edge_length_list = []
fig = Figure()
for i = 1:5

    mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    println("i = ", i)
    println("nCells: ", mesh.cells.n)
    println("nEdges: ", mesh.edges.n)
    println("nVertices: ", mesh.vertices.n)
    xp = mesh.x_period
    yp = mesh.y_period
 
    field = x -> [sin(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp), cos(4*pi*x[1]/xp) * sin(4*pi*x[2]/yp)]
    true_divergence = x -> 8*pi*cos(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp)

    u_cochain = discretize_vector_field_dual(mesh, field)
    div_cochain = Cochain(true_divergence.(mesh.cells.centroid) .* mesh.cells.area, 2, false)

    div_discrete = divergence_diagonal(mesh, u_cochain)

    push!(error_list_diagonal, maximum(abs.(div_discrete.values - div_cochain.values)))
    #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
    push!(edge_length_list, maximum(mesh.edges.lengthDual))
    println("--------------------------------")

    if i == 5
        diference = Cochain(div_discrete.values - div_cochain.values, 2, false)
        fig = Figure()
        ax = Axis(fig[1, 1], title = "Divergence error with diagonal Hodge star on regular mesh")
        fig, ax = plot_cochain(fig, ax, diference, mesh)
        CairoMakie.save("results/laplacian/divergence_error_diagonal_regular.pdf", fig)
    end
end

println("Error list diagonal: ", error_list_diagonal)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_diagonal)
    order = log(error_list_diagonal[i]/error_list_diagonal[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end

println("TESTE COM MALHA IRREGULAR")
error_list_diagonal = []
edge_length_list = []
fig = Figure()
for i = 1:5

    mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    println("i = ", i)
    println("nCells: ", mesh.cells.n)
    println("nEdges: ", mesh.edges.n)
    println("nVertices: ", mesh.vertices.n)
    xp = mesh.x_period
    yp = mesh.y_period

    field = x -> [sin(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp), cos(4*pi*x[1]/xp) * sin(4*pi*x[2]/yp)]
    true_divergence = x -> 8*pi*cos(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp)

    u_cochain = discretize_vector_field_dual(mesh, field)
    div_cochain = Cochain(true_divergence.(mesh.cells.centroid) .* mesh.cells.area, 2, false)

    div_discrete = divergence_diagonal(mesh, u_cochain)

    push!(error_list_diagonal, maximum(abs.(div_discrete.values - div_cochain.values)))
    #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
    push!(edge_length_list, maximum(mesh.edges.lengthDual))
    println("--------------------------------")

    if i == 5
        diference = Cochain(div_discrete.values - div_cochain.values, 2, false)
        fig = Figure()
        ax = Axis(fig[1, 1], title = "Divergence error with diagonal Hodge star on irregular mesh")
        fig, ax = plot_cochain(fig, ax, diference, mesh)
        CairoMakie.save("results/laplacian/divergence_error_diagonal_irregular.pdf", fig)
    end
end

println("Error list diagonal: ", error_list_diagonal)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_diagonal)
    order = log(error_list_diagonal[i]/error_list_diagonal[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end


In [ ]:
function divergence_lsq(mesh::VoronoiMesh, u::Cochain{T}, linear::Bool = true) where T
    if u.k != 1 || !u.is_primal
        error("Input must be a primal 1-form")
    end
    d1 = d1_matrix_dual(mesh)
    star_u = hodge_star_lsq_matrix(mesh, true, linear)
    return Cochain(d1 * star_u * u.values, 2, false)
end

error_list_diagonal = []
edge_length_list = []
fig = Figure()
println("TESTE COM MALHA REGULAR")
for (j, linear) in enumerate([true, false])
    for i = 1:5

        mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
        println("i = ", i)
        println("nCells: ", mesh.cells.n)
        println("nEdges: ", mesh.edges.n)
        println("nVertices: ", mesh.vertices.n)
        xp = mesh.x_period
        yp = mesh.y_period
    
        field = x -> [sin(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp), cos(4*pi*x[1]/xp) * sin(4*pi*x[2]/yp)]
        true_divergence = x -> 8*pi*cos(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp)

        u_cochain = discretize_vector_field_dual(mesh, field)
        div_cochain = Cochain(true_divergence.(mesh.cells.centroid) .* mesh.cells.area, 2, false)

        div_discrete = divergence_lsq(mesh, u_cochain, linear)

        push!(error_list_diagonal, maximum(abs.(div_discrete.values - div_cochain.values)))
        #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
        push!(edge_length_list, maximum(mesh.edges.lengthDual))
        println("--------------------------------")

        if i == 5
            diference = Cochain(div_discrete.values - div_cochain.values, 2, false)
            fig = Figure()
            ax = Axis(fig[1, 1], title = "Divergence error with LSQ-$j Hodge star on regular mesh")
            fig, ax = plot_cochain(fig, ax, diference, mesh)
                CairoMakie.save("results/laplacian/divergence_error_lsq$(j)_regular.pdf", fig)
        end
    end
end

println("Error list diagonal: ", error_list_diagonal)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_diagonal)
    order = log(error_list_diagonal[i]/error_list_diagonal[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end

println("TESTE COM MALHA IRREGULAR")
error_list_diagonal = []
edge_length_list = []
fig = Figure()
for (j, linear) in enumerate([true, false])
    for i = 1:5

        mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
        println("i = ", i)
        println("nCells: ", mesh.cells.n)
        println("nEdges: ", mesh.edges.n)
        println("nVertices: ", mesh.vertices.n)
        xp = mesh.x_period
        yp = mesh.y_period

        field = x -> [sin(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp), cos(4*pi*x[1]/xp) * sin(4*pi*x[2]/yp)]
        true_divergence = x -> 8*pi*cos(4*pi*x[1]/xp) * cos(4*pi*x[2]/yp)

        u_cochain = discretize_vector_field_dual(mesh, field)
        div_cochain = Cochain(true_divergence.(mesh.cells.centroid) .* mesh.cells.area, 2, false)

        div_discrete = divergence_lsq(mesh, u_cochain, linear)

        push!(error_list_diagonal, maximum(abs.(div_discrete.values - div_cochain.values)))
        #push!(error_list_diagonal, norm((lap_diagonal.values - lap_cochain.values) .* sqrt.(mesh.cells.area)))
        push!(edge_length_list, maximum(mesh.edges.lengthDual))
        println("--------------------------------")

        if i == 5
            diference = Cochain(div_discrete.values - div_cochain.values, 2, false)
            fig = Figure()
            ax = Axis(fig[1, 1], title = "Divergence error with LSQ-$j Hodge star on irregular mesh")
            fig, ax = plot_cochain(fig, ax, diference, mesh)
            CairoMakie.save("results/laplacian/divergence_error_lsq$(j)_irregular.pdf", fig)
        end
    end
end

println("Error list diagonal: ", error_list_diagonal)
println("Edge length list: ", edge_length_list)
for i = 2:length(error_list_diagonal)
    order = log(error_list_diagonal[i]/error_list_diagonal[i-1]) / log(edge_length_list[i]/edge_length_list[i-1])
    println("Order Linf between meshes $(i-1) and $i: $order")
end



In [ ]:
mesh = VoronoiMesh("meshes/mesh3.nc")
d0 = d0_matrix_primal(mesh)
d1 = d1_matrix_dual(mesh)
star1_diagonal = star(mesh, 1, true)
star1_lsq1 = hodge_star_lsq_matrix(mesh, true, true)
star1_lsq2 = hodge_star_lsq_matrix(mesh, true, false)
G1 = d1 * star1_diagonal  * d0
G2 = d1 * star1_lsq1  * d0
G3 = d1 * star1_lsq2  * d0


In [ ]:
using LinearAlgebra
eig1 = eigvals(Matrix(G1))
eig2 = eigvals(Matrix(G2))
eig3 = eigvals(Matrix(G3))
fig = Figure()
ax = Axis(fig[1, 1], title = "Eigenvalues of G1")
scatter!(ax, eig1)
fig

In [ ]:
fig = Figure(size = (900, 400))
ax1 = Axis(fig[1, 1], title = L"Eigenvalues of $\Delta_{diag}$", xlabel = L"Re(\lambda)", ylabel = L"Im(\lambda)")
for i in 1:length(eig1)
    scatter!(ax1, real(eig1[i]), imag(eig1[i]), color = :red)
end
ax2 = Axis(fig[1, 2], title = L"Eigenvalues of $\Delta_{lsq1}$", xlabel = L"Re(\lambda)", ylabel = L"Im(\lambda)")
for i in 1:length(eig2)
    scatter!(ax2, real(eig2[i]), imag(eig2[i]), color = :blue)
end
ax3 = Axis(fig[1, 3], title = L"Eigenvalues of $\Delta_{lsq2}$", xlabel = L"Re(\lambda)", ylabel = L"Im(\lambda)")
for i in 1:length(eig3)
    scatter!(ax3, real(eig3[i]), imag(eig3[i]), color = :green)
end
CairoMakie.save("results/laplacian/eigenvalues.pdf", fig)

In [ ]:
eig3

In [ ]:
star(mesh, 2, false)